<a href="https://colab.research.google.com/github/sanaisrail/urdu-ocr-codesaviours-si26-Sana/blob/main/week3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install transformers torch pillow pandas

In [3]:
!pip install sentencepiece

In [4]:
!pip uninstall -y tokenizers transformers sentencepiece
!pip install transformers==4.38.2 tokenizers==0.15.2 sentencepiece protobuf

Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
Found existing installation: transformers 5.13.1
Uninstalling transformers-5.13.1:
  Successfully uninstalled transformers-5.13.1
Found existing installation: sentencepiece 0.2.2
Uninstalling sentencepiece-0.2.2:
  Successfully uninstalled sentencepiece-0.2.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 114.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.8 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.23.0
    Uninstalling huggingface_hub-1.23.0:
      Successfully uninstalled huggingface_hub-1.23.0
ERROR: pip's dependency resolve

In [1]:
from transformers import TrOCRProcessor

processor = TrOCRProcessor.from_pretrained(
    "microsoft/trocr-base-printed"
)

print("Processor loaded")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Processor loaded


In [3]:
import pandas as pd
import os

df = pd.read_csv("/content/drive/MyDrive/dataset (1).csv")

df["image"] = df["image"].apply(
    lambda x: "/content/drive/MyDrive/data" + os.path.basename(x)
)

df.to_csv("/content/drive/MyDrive/dataset (1).csv", index=False)

print("labels_fixed.csv created successfully")

labels_fixed.csv created successfully


In [4]:
csv_path = "/content/drive/MyDrive/dataset (1).csv"

In [6]:
!pip install transformers torch pillow pandas

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor
from PIL import Image
import pandas as pd


class UrduOCRDataset(Dataset):

    def __init__(self, csv_path, processor):
        self.data = pd.read_csv("/content/drive/MyDrive/dataset (1).csv")
        self.processor = processor
        print(f"Dataset loaded: {len(self.data)} samples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        # Load and convert image
        image = Image.open(row['image']).convert("RGB")

        # Process image for the model
        encoding = self.processor(
            image,
            return_tensors="pt"
        )
        pixel_values = encoding.pixel_values.squeeze()

        # Process the text label
        labels = self.processor.tokenizer(
            row['text'],
            padding="max_length",
            max_length=128
        ).input_ids

        labels = torch.tensor(labels)

        return {
            "pixel_values": pixel_values,
            "labels": labels
        }

In [44]:
# Load the TrOCR processor
processor = TrOCRProcessor.from_pretrained(
    "microsoft/trocr-base-handwritten",
    use_fast=False
)

# Create dataset
dataset = UrduOCRDataset("/content/drive/MyDrive/dataset (1).csv", processor)

# Test it loads correctly
sample = dataset[0]

print("Sample pixel_values shape:", sample["pixel_values"].shape)
print("Sample labels shape:", sample["labels"].shape)
print("Dataset is working correctly!")

# Create train / test split (80% train, 20% test)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = torch.utils.data.random_split(
    dataset,
    [train_size, test_size]
)

print(f"Training samples: {train_size}")
print(f"Testing samples: {test_size}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Dataset loaded: 200 samples
Sample pixel_values shape: torch.Size([3, 384, 384])
Sample labels shape: torch.Size([179])
Dataset is working correctly!
Training samples: 160
Testing samples: 40


In [28]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/dataset (1).csv")

print(df.head())
print(df.columns)

                               image  \
0  /content/drive/MyDrive/image1.png   
1  /content/drive/MyDrive/image2.png   
2  /content/drive/MyDrive/image3.png   
3  /content/drive/MyDrive/image4.png   
4  /content/drive/MyDrive/image5.png   

                                                text  
0  ایران کے گلستان صوبے میں واقع آقتکہ خان ریلوے ...  
1  روان سال فروری میں، ایران کے خلاف امریکہ اور ا...  
2  اس پل کی اہمیت کو دو اہم بین الاقوامی راستوں ک...  
3  ایرانی ریلوے حکام کے مطابق، گذشتہ سال کم از کم...  
4         پٹرول سے سیرامکس تک تجارت کا ایک اہم راستہ  
Index(['image', 'text'], dtype='object')


In [41]:
import pandas as pd
import os

csv_path = "/content/drive/MyDrive/dataset (1).csv"

df = pd.read_csv(csv_path)

image_root = "/content/drive/MyDrive/data"

# Saari images collect karo
image_paths = []

for root, dirs, files in os.walk(image_root):
    for file in files:
        if file.lower().endswith((".png", ".jpg", ".jpeg")):
            image_paths.append(os.path.join(root, file))

print("Total images:", len(image_paths))
print(image_paths[:5])

# CSV ke image paths replace
df["image"] = image_paths[:len(df)]

# Save
df.to_csv(csv_path, index=False)

print(df.head())

Total images: 200
['/content/drive/MyDrive/data/other/1.png', '/content/drive/MyDrive/data/other/20.png', '/content/drive/MyDrive/data/other/5.png', '/content/drive/MyDrive/data/other/35.png', '/content/drive/MyDrive/data/other/34.png']
                                      image  \
0   /content/drive/MyDrive/data/other/1.png   
1  /content/drive/MyDrive/data/other/20.png   
2   /content/drive/MyDrive/data/other/5.png   
3  /content/drive/MyDrive/data/other/35.png   
4  /content/drive/MyDrive/data/other/34.png   

                                                text  
0  ایران کے گلستان صوبے میں واقع آقتکہ خان ریلوے ...  
1  روان سال فروری میں، ایران کے خلاف امریکہ اور ا...  
2  اس پل کی اہمیت کو دو اہم بین الاقوامی راستوں ک...  
3  ایرانی ریلوے حکام کے مطابق، گذشتہ سال کم از کم...  
4         پٹرول سے سیرامکس تک تجارت کا ایک اہم راستہ  
